# Evidence-aware v0–v4 comparison

**Goal:** Separate inference-only preprocessing, matched training and joint effects using the 13 permitted Raw-vs-policy cells. A = Raw/Raw, B = Raw/p, C = p/Raw, D = p/p. No policy-to-policy cross-input evaluations.

**Inputs:** Saved run config snapshots, development `metrics.json`, training completion markers and optional native histories/statistics; `configs/comparison_evidence.json` supplies archive hashes, runtime sources and frozen split/GT attestations.

**Outputs:** In-memory evidence tables, deterministic figures and an evidence-generated summary. Opt-in inference writes missing evaluations through the existing validated evaluation path only.

**Prerequisites:** Install the project kernel. Default report mode needs saved artifacts and frozen split metadata, with no GPU, weights, image exports or original pixels. Missing evidence stays visible. No training, re-export or sealed final-holdout access.


<a id="section-1"></a>

## 1. Configuration

The defaults pin the published runs; change a date explicitly, or use `None` to select the latest run. Unindexed runs remain descriptive-only until their protocol/runtime evidence is recorded. Archive hashes validate saved files; they do not revalidate original exported pixels or boxes.

Historical batch-32 v0–v3 and matched batch-16 Raw/v4 are separate studies. The archived `v5_raw-2026-11-11` alias is **not** the shared `v0/v0` arm. It cannot complete v4's historical 2×2.


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.comparison import (
    POLICIES,
    benchmark_preprocessing,
    compare_policies,
    evaluate_matrix,
    evidence_summary,
    inspect_report,
    policy_cohorts,
    result_table,
    training_histories,
)
from vehicle_detection.comparison_plots import (
    plot_classes,
    plot_cohorts,
    plot_convergence,
    plot_cost,
    plot_coverage,
    plot_effects,
    plot_factorial,
    plot_overall,
)

ROOT = project_root()
os.chdir(ROOT)
require_paths(protocol=ROOT / "configs/data.yaml")

RUN_DATES = {
    "v0": "2026-11-09",
    "v1": "2026-11-09",
    "v2": "2026-11-09",
    "v3": "2026-11-09",
    "v4": "2026-11-10",
}
ALLOW_INFERENCE = False
INFERENCE_POLICIES = ("v1", "v2", "v3")  # One compatible profile per inference call
BOOTSTRAP_RESAMPLES = 0  # 500+ enables paired, source-group 95% percentile CIs
BOOTSTRAP_SEED = 2026
MEASURE_PREPROCESS_COST = False


def show(figure):
    display(figure)
    plt.close(figure)

<a id="section-2"></a>

## 2. Setup & Provenance / Coverage

All 13 permitted pairs are shown, including missing, blocked, stale and incomplete evidence. Saved metrics can be read without `best.pt` or `data/data.yaml`. An absent optional NPZ disables uncertainty; a present NPZ with a changed hash or development membership invalidates its cell. Stale artifacts are never overwritten or silently recomputed.

`baseline_eligible` requires an indexed frozen protocol and runtime, matching training recipe, initialization, seed and evaluation settings. The archive attests GT geometry; report mode does not independently recheck exported labels. Full checkpoint hashes and metric paths/hashes below identify every diagonal. Cross-cache identities bind model checkpoint, input files/metadata, runtime and settings; the index additionally pins their saved bytes.


In [ ]:
report = inspect_report(ROOT, run_dates=RUN_DATES)
display(pd.DataFrame(report["provenance"]))
overview = result_table(report)
display(overview)
show(plot_coverage(report))
print(
    "Archived matched control status:",
    report["blocked"].get(
        "matched_raw", "available" if "matched_raw" in report["contexts"] else "missing"
    ),
)

<a id="section-3"></a>

## 3. Optional Evaluation

`ALLOW_INFERENCE=False` keeps report mode read-only. To compute missing cells, explicitly enable inference and select one compatible profile via `INFERENCE_POLICIES`. This uses the original export-byte, selected-checkpoint, membership, leakage and cache-identity gates; report evidence cannot authorize inference. Batch-16 v4 cannot be evaluated against historical batch-32 v0 through this interface. No training or export is started.

New caches require a reviewed provenance-index update before becoming report comparison evidence. CPU policy benchmarking is separately gated and uses only the validated exported development Raw JPEG proxies.


In [ ]:
cpu_cost = None
if ALLOW_INFERENCE:
    evaluation = evaluate_matrix(
        ROOT, run_dates=RUN_DATES, policies=INFERENCE_POLICIES, allow_inference=True
    )
    display(result_table(evaluation))
    report = inspect_report(ROOT, run_dates=RUN_DATES)
    overview = result_table(report)
if MEASURE_PREPROCESS_COST:
    from vehicle_detection.comparison import inspect_matrix

    timing_context = inspect_matrix(ROOT, run_dates=RUN_DATES, policies=INFERENCE_POLICIES)
    cpu_cost = benchmark_preprocessing(timing_context, seed=BOOTSTRAP_SEED)

<a id="section-4"></a>

## 4. Overall Accuracy

Absolute mAP50, mAP50:95, precision and recall are percentages. Deltas use **percentage points (pp)** against the named same-profile Raw baseline. Each panel is a separate study, never a combined batch-16/batch-32 leaderboard. Missing or unverified baselines remain in coverage/provenance rather than producing an unmatched delta.


In [ ]:
show(plot_overall(report))

<a id="section-5"></a>

## 5. Factorial Effects & Uncertainty

Complete A/B/C/D is required for all six contrasts: B−A (inference on Raw model), D−B (training at processed input), C−A (training at Raw input), D−C (inference on policy model), D−A (joint), and (D−C)−(B−A) (interaction).

Bootstrap is opt-in and needs valid per-image NPZ statistics plus frozen development source-group metadata. The same group draws are shared by all four arms. **CI unavailable** means no interval was computed, not zero uncertainty. A CI crossing zero is inconclusive, not proof of equivalence. Intervals are conditional on fixed single-seed checkpoints and do not adjust for multiple comparisons or checkpoint selection.


In [ ]:
effects, intervals = compare_policies(
    report, bootstrap_resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED
)
display(effects)
show(plot_factorial(report))
show(plot_effects(report, effects))

<a id="section-6"></a>

## 6. Per-class & Fixed Cohorts

Class heatmaps use matched Raw baselines and zero-centered pp scales; absent/zero GT support is N/A. Motorcycle and bicycle are highlighted to make two-wheeler regressions visible.

Cohort comparisons require validated policy metadata with **identical development image IDs** in every arm: raw brightness <0.12 for Dark, otherwise Bright; policy routing defines Apply/Skip. Matching image counts alone does not prove membership. Missing routing metadata or paired statistics suppresses comparison, and missing cohorts are never AP=0. Cohort plots show image and GT-instance support when available.


In [ ]:
show(plot_classes(report))
cohorts = {policy: policy_cohorts(report, policy=policy) for policy in POLICIES}
show(plot_cohorts(report, cohorts))

<a id="section-7"></a>

## 7. Convergence & Cost

Saved training histories show calibration AP and available training/validation losses, progress toward 50 epochs, and the peak rounded CSV AP50:95 epoch (an inferred selector, not a checkpoint-deserialized epoch). Histories and native args must agree with their config and completion markers. Batch-16 and batch-32 curves and cost panels remain separate.

Training minutes are cumulative native history time. Native preprocessing/inference/postprocessing timings exclude external policy processing. Optional CPU mean/p50/p95 uses one OpenCV thread on decoded Raw JPEG95 proxies, excludes decoding/encoding/I/O/detector costs, and is **never summed** with GPU timings. Offline preparation is not online latency; device/runtime differences and uncontrolled cache/order/thermal effects preclude a hardware speedup claim.


In [ ]:
histories, history_unavailable = training_histories(report)
print("Unavailable histories:", history_unavailable or "none")
show(plot_convergence(report, histories))
show(plot_cost(report, histories, cpu_cost))

<a id="section-8"></a>

## 8. Qualitative Error Gallery

Gallery unavailable with the published report artifacts: per-image matching statistics contain no prediction boxes, and the archive has no paired, compatible Raw/policy prediction exports plus development pixels. To enable a reproducible GT / Raw / policy gallery, supply validated box/score/class predictions and GT geometry for explicit fixed development IDs in the same profile. No inference or pixel access is triggered here. Visual samples are diagnostic illustrations, never independent statistical evidence.


<a id="section-9"></a>

## 9. Evidence-backed Summary

This summary is generated from the loaded cells and matched baselines. Incomplete or incompatible designs cannot answer factorial questions. Point estimates describe the available single-seed development evidence; negative or inconclusive results do not establish mechanisms. The final holdout remains sealed.


In [ ]:
for line in evidence_summary(report, effects, intervals=intervals):
    print(line)
print("Saved cell statuses:", overview.status.value_counts().to_dict())
print("No training or data re-export. Inference enabled:", ALLOW_INFERENCE)